<a href="https://colab.research.google.com/github/mariaaapetrovskaya/complingua/blob/main/gesture_dataset_creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets huggingface-hub pandas gdown -q

from google.colab import drive
drive.mount('/content/drive')


from google.colab import files
uploaded = files.upload()

import pandas as pd
import os
import re
import shutil
from huggingface_hub import HfApi, login
from google.colab import userdata

df = pd.read_csv('gestures_dataset.csv')


df = df.rename(columns={
    'folder/video_gesture.mp4': 'video_link',
    'название жеста/описание': 'gesture_description',
    'значение жеста': 'gesture_value',
    'тип жеста': 'gesture_type'
})

print("Колонки:", df.columns.tolist())
print(df.head())

video_base = '/content/drive/MyDrive/gesture_videos/'

def get_local_path(link):
    """Извлекает file_id из ссылки или возвращает имя файла, если это не ссылка."""
    if pd.isna(link) or not isinstance(link, str):
        return None
    match = re.search(r'/d/([a-zA-Z0-9_-]+)', link)
    if match:
        file_id = match.group(1)
        return os.path.join(video_base, f"{file_id}.mp4")
    else:

        return os.path.join(video_base, link)

df['video_path'] = df['video_link'].apply(get_local_path)

df['exists'] = df['video_path'].apply(lambda p: os.path.exists(p) if p else False)
print("\nСтатистика наличия видео:")
print(df['exists'].value_counts())


df = df[df['exists']].copy()
print(f"Осталось видео: {len(df)}")

root_dir = '/content/videofolder_dataset'
os.makedirs(root_dir, exist_ok=True)

unique_types = df['gesture_type'].unique()
print(f"Найдены типы жестов: {unique_types}")

for gesture_type in unique_types:

    safe_name = gesture_type.replace(' ', '_').replace('/', '_').replace('-', '_')
    class_dir = os.path.join(root_dir, safe_name)
    os.makedirs(class_dir, exist_ok=True)

    subset = df[df['gesture_type'] == gesture_type]
    for _, row in subset.iterrows():
        src = row['video_path']
        dst = os.path.join(class_dir, os.path.basename(src))
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            print(f"Копирую {os.path.basename(src)} в {safe_name}")
        else:
            print(f"Файл {os.path.basename(src)} уже есть в {safe_name}")

print("\n VideoFolder создана во временной папке.")


HF_TOKEN = userdata.get('HF_TOKEN')
if HF_TOKEN is None:
    raise ValueError("Токен HF_TOKEN не найден. Добавьте его")
login(token=HF_TOKEN)

api = HfApi()
repo_id = "mapetrovska/gesturedataset"

print("\n=== Загружаем датасет на Hugging Face ===")
api.upload_folder(
    folder_path=root_dir,
    path_in_repo="",
    repo_id=repo_id,
    repo_type="dataset",
    token=HF_TOKEN
)

print(f"\n✅ Датасет успешно загружен! https://huggingface.co/datasets/{repo_id}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving gestures_dataset - Лист1.csv to gestures_dataset - Лист1.csv
Колонки: ['video_link', 'gesture_description', 'gesture_value', 'gesture_type']
                                          video_link gesture_description  \
0  https://drive.google.com/file/d/1-yQJpvbh21sP0...    показать на себя   
1  https://drive.google.com/file/d/1zScjsQ9W9TjZC...      показать рукой   
2  https://drive.google.com/file/d/1kZ_OQx8Kcg_Yb...  показывать на себя   
3  https://drive.google.com/file/d/1XRDzNlqjG8227...      ткунть пальцем   
4  https://drive.google.com/file/d/1Se775d9nV7sIR...  показывать на себя   

       gesture_value        gesture_type  
0  самоидентификация  дейктические жесты  
1  общеуказательный   дейктические жесты  
2  самоидентификация  дейктические жесты  
3   общеуказательный  дейктические жесты  
4  самоидентификация  дейктические жесты  

Статистика наличия видео:
exists
True    102
Name: count, dtype: int64
Осталось видео: 102
Найдены типы жестов: ['дейктические жесты' 'д

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pu8pIrAXCy_PQ2J_nYLpR.mp4: 100%|##########|  299kB /  299kB            

...

  ...jzE6mScl-ZGxkEpFKh0M-.mp4: 100%|##########|  310kB /  310kB            

  ...YbrWU8VFyzvj0MBwKAk-F.mp4: 100%|##########| 2.57MB / 2.57MB            

  ...y8YtmNLX_PKVgUsVJno77.mp4: 100%|##########|  887kB /  887kB            

  ...Om55ifLgfYp2ayYiqJ_lC.mp4: 100%|##########| 1.31MB / 1.31MB            


✅ Датасет успешно загружен! https://huggingface.co/datasets/mapetrovska/gesturedataset
